In [1]:
# check_checkpoint.py
import torch
import os
import pickle

def diagnose_checkpoint(filepath):
    print(f"Diagnosing checkpoint: {filepath}")
    print(f"File size: {os.path.getsize(filepath)} bytes")
    
    try:
        # 尝试直接加载
        checkpoint = torch.load(filepath, map_location='cpu')
        print("✓ Successfully loaded with torch.load")
        print("Keys in checkpoint:", checkpoint.keys())
        return checkpoint
    except Exception as e:
        print(f"✗ torch.load failed: {e}")
    
    try:
        # 尝试用pickle加载
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
        print("✓ Successfully loaded with pickle")
        print("Type:", type(data))
        return data
    except Exception as e:
        print(f"✗ pickle.load failed: {e}")
    
    try:
        # 尝试读取原始字节
        with open(filepath, 'rb') as f:
            raw_data = f.read()
        print(f"Raw data length: {len(raw_data)} bytes")
        print(f"First 100 bytes: {raw_data[:100]}")
        return None
    except Exception as e:
        print(f"✗ Even raw read failed: {e}")
    
    return None

# 运行诊断
filepath = "outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251026_015146/checkpoint_epoch_0000_20251027_103724.pkl"
checkpoint = diagnose_checkpoint(filepath)

Diagnosing checkpoint: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251026_015146/checkpoint_epoch_0000_20251027_103724.pkl
File size: 594297 bytes
✗ torch.load failed: Weights only load failed. In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
Please file an issue with the following so that we can make `weights_only=True` compatible with your use case: WeightsUnpickler error: Unsupported operand 149

Check the documentation of torch.load to learn more about types accepted by default with weights_only https://pytorch.org/docs/stable/generated/torch.load.html.
✓ Successfully loaded with pickle
Type: <class 'dict'>


/usr/local/lib64/python3.12/site-packages/torch/_weights_only_unpickler.py:549: UserWarning: Detected pickle protocol 4 in the checkpoint, which was not the default pickle protocol used by `torch.load` (2). The weights_only Unpickler might not support all instructions implemented by this protocol, please file an issue for adding support if you encounter this.
  warnings.warn(


In [ ]:
# inspect_checkpoint.py
import torch
import pickle
import json
import os
import sys

def inspect_checkpoint(filepath):
    print(f"Inspecting checkpoint: {filepath}")
    
    # 检查文件是否存在
    if not os.path.exists(filepath):
        print(f"❌ Checkpoint file does not exist: {filepath}")
        return None
    
    print(f"File size: {os.path.getsize(filepath)} bytes")
    
    # 方法1: 使用pickle
    try:
        with open(filepath, 'rb') as f:
            checkpoint = pickle.load(f)
        
        print("✅ Successfully loaded with pickle")
        print("Checkpoint keys:", checkpoint.keys())
        print(f"Epoch: {checkpoint.get('epoch', 'N/A')}")
        print(f"Best accuracy: {checkpoint.get('best_acc', 'N/A')}")
        print(f"Best validation accuracy: {checkpoint.get('best_val_accuracy', 'N/A')}")
        
        if 'model_state_dict' in checkpoint:
            print(f"Model state dict keys: {len(checkpoint['model_state_dict'])}")
            # 打印前几个键
            for i, key in enumerate(list(checkpoint['model_state_dict'].keys())[:5]):
                tensor = checkpoint['model_state_dict'][key]
                if hasattr(tensor, 'shape'):
                    print(f"  {i}: {key} - {tensor.shape}")
                else:
                    print(f"  {i}: {key} - no shape info")
        
        if 'optimizer_state_dict' in checkpoint:
            print("✅ Optimizer state dict available")
        
        if 'scheduler_state_dict' in checkpoint:
            print("✅ Scheduler state dict available")
        
        if 'training_stats' in checkpoint:
            print("Training stats available")
        
        if 'metrics_history' in checkpoint:
            print("Metrics history available")
            
        return checkpoint
        
    except Exception as e:
        print(f"❌ Failed to inspect checkpoint with pickle: {e}")
        
        # 尝试使用 torch.load
        try:
            checkpoint = torch.load(filepath, map_location='cpu', weights_only=False)
            print("✅ Successfully loaded with torch.load (weights_only=False)")
            print("Checkpoint keys:", checkpoint.keys())
            return checkpoint
        except Exception as e2:
            print(f"❌ Failed to load with torch.load: {e2}")
            return None

def main():
    if len(sys.argv) > 1:
        filepath = sys.argv[1]
        # 检查是否是参数标志
        if filepath.startswith('-'):
            print("Usage: python inspect_checkpoint.py [checkpoint_path]")
            print("If no path provided, uses default path")
            default_path = "outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251026_015146/checkpoint_epoch_0000_20251027_103724.pkl"
            print(f"Using default path: {default_path}")
            filepath = default_path
    else:
        filepath = "outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl"
    
    checkpoint = inspect_checkpoint(filepath)
    
    if checkpoint is not None:
        print("\n" + "="*50)
        print("CHECKPOINT SUMMARY")
        print("="*50)
        print(f"Epoch: {checkpoint.get('epoch', 'Unknown')}")
        print(f"Best accuracy: {checkpoint.get('best_acc', checkpoint.get('best_val_accuracy', 'Unknown'))}")
        
        # 建议下一步操作
        print("\n" + "="*50)
        print("NEXT STEPS")
        print("="*50)
        print("If checkpoint looks good, run:")
        print(f"python train_chaotic.py --system lorenz --epochs 100 --save_models --model_types full_chaotic --data_dir /scratch/project_2003370/yueyao/dataset/train-clean-100/LibriSpeech/train-clean-100 --resume {filepath}")

if __name__ == "__main__":
    main()

In [6]:
# convert_checkpoint.py
import torch
import pickle
import os
import sys

def convert_checkpoint(old_path, new_path):
    """转换检查点格式"""
    try:
        # 检查原文件是否存在
        if not os.path.exists(old_path):
            print(f"❌ Original checkpoint not found: {old_path}")
            return False
            
        # 用pickle加载
        with open(old_path, 'rb') as f:
            checkpoint = pickle.load(f)
        
        print(f"✅ Successfully loaded original checkpoint")
        print(f"Keys: {checkpoint.keys()}")
        
        # 保存为torch格式（使用较低的pickle协议提高兼容性）
        torch.save(checkpoint, new_path, pickle_protocol=2)
        print(f"✅ Converted {old_path} to {new_path}")
        
        # 验证转换后的文件
        if os.path.exists(new_path):
            test_checkpoint = torch.load(new_path, map_location='cpu')
            print(f"✅ Conversion verified - new file contains keys: {test_checkpoint.keys()}")
            return True
        else:
            print(f"❌ Converted file not created")
            return False
            
    except Exception as e:
        print(f"❌ Conversion failed: {e}")
        return False

def main():
    if len(sys.argv) > 1:
        old_path = sys.argv[1]
        # 检查是否是参数标志
        if old_path.startswith('-'):
            print("Usage: python convert_checkpoint.py [old_checkpoint_path] [new_checkpoint_path]")
            print("If no paths provided, uses default paths")
            default_old = "outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl"
            default_new = default_old.replace('.pkl', '_converted.pth')
            print(f"Using default paths:")
            print(f"  Old: {default_old}")
            print(f"  New: {default_new}")
            old_path = default_old
            new_path = default_new
        else:
            if len(sys.argv) > 2:
                new_path = sys.argv[2]
            else:
                new_path = old_path.replace('.pkl', '_converted.pth')
    else:
        old_path = "outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl"
        new_path = old_path.replace('.pkl', '_converted.pth')
    
    success = convert_checkpoint(old_path, new_path)
    
    if success:
        print(f"\n✅ Conversion successful!")
        print(f"Original: {old_path}")
        print(f"Converted: {new_path}")
        print(f"\nNow you can use the converted checkpoint:")
        print(f"python train_chaotic.py ... --resume {new_path}")
    else:
        print(f"\n❌ Conversion failed!")

if __name__ == "__main__":
    main()

Usage: python convert_checkpoint.py [old_checkpoint_path] [new_checkpoint_path]
If no paths provided, uses default paths
Using default paths:
  Old: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl
  New: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020_converted.pth
✅ Successfully loaded original checkpoint
Keys: dict_keys(['epoch', 'timestamp', 'metrics', 'additional_info', 'model', 'framework'])
✅ Converted outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl to outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020_converted.pth
✅ Conversion verified - new file contains keys: dict_keys(['epoch', 'timestamp', 'metrics', 'additional_info', 'model', 'framework'

In [2]:
# debug_resume.py
import sys
import os

def debug_resume_param():
    """调试恢复参数传递"""
    print("=== Debug Resume Parameter ===")
    
    # 检查命令行参数
    print("Command line arguments:", sys.argv)
    
    # 检查是否有 --resume 参数
    if '--resume' in sys.argv:
        resume_index = sys.argv.index('--resume')
        if resume_index + 1 < len(sys.argv):
            resume_path = sys.argv[resume_index + 1]
            print(f"Found --resume parameter: {resume_path}")
            
            # 检查文件是否存在
            if os.path.exists(resume_path):
                print(f"✓ Checkpoint file exists: {resume_path}")
                print(f"File size: {os.path.getsize(resume_path)} bytes")
            else:
                print(f"✗ Checkpoint file not found: {resume_path}")
        else:
            print("✗ --resume parameter provided but no path specified")
    else:
        print("✗ No --resume parameter found in command line")

if __name__ == "__main__":
    debug_resume_param()

=== Debug Resume Parameter ===
Command line arguments: ['/usr/local/lib/python3.12/site-packages/ipykernel_launcher.py', '-f', '/users/tianyuey/.local/share/jupyter/runtime/kernel-c71e3c17-8c25-4a86-a5b6-78d4d49c910d.json']
✗ No --resume parameter found in command line


In [4]:
!python debug_resume.py --resume outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl    

=== Debug Resume Parameter ===
Command line arguments: ['debug_resume.py', '--resume', 'outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl']
Found --resume parameter: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl
✓ Checkpoint file exists: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl
File size: 592939 bytes


In [6]:
!python debug_resume.py --resume outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251026_015146/checkpoint_epoch_0000_20251027_103724.pkl

=== Debug Resume Parameter ===
Command line arguments: ['debug_resume.py', '--resume', 'outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251026_015146/checkpoint_epoch_0000_20251027_103724.pkl']
Found --resume parameter: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251026_015146/checkpoint_epoch_0000_20251027_103724.pkl
✓ Checkpoint file exists: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251026_015146/checkpoint_epoch_0000_20251027_103724.pkl
File size: 594297 bytes


In [8]:
# find_hardcoded_paths.py
import os
import re

def find_hardcoded_paths(filepath):
    """查找文件中的硬编码路径"""
    with open(filepath, 'r') as f:
        content = f.read()
    
    # 查找可能的硬编码路径模式
    patterns = [
        r'exp_20251026_015146',
        r'checkpoint_epoch_0000_20251027_103724\.pkl',
        r'outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints'
    ]
    
    found = False
    for pattern in patterns:
        matches = re.findall(pattern, content)
        if matches:
            print(f"Found hardcoded path pattern: {pattern}")
            print(f"Matches: {matches}")
            found = True
    
    if not found:
        print("No hardcoded paths found")
    
    return found

if __name__ == "__main__":
    filepath = "train_chaotic.py"
    if os.path.exists(filepath):
        find_hardcoded_paths(filepath)
    else:
        print(f"File not found: {filepath}")

No hardcoded paths found


In [10]:
# analyze_checkpoint_fixed.py
import torch
import pickle
import os
import sys

def analyze_checkpoint(filepath):
    print(f"Analyzing checkpoint: {filepath}")
    
    # 检查文件是否存在
    if not os.path.exists(filepath):
        print(f"❌ Checkpoint file does not exist: {filepath}")
        return None
    
    print(f"File size: {os.path.getsize(filepath)} bytes")
    
    try:
        # 使用pickle加载
        with open(filepath, 'rb') as f:
            checkpoint = pickle.load(f)
        
        print("✅ Successfully loaded with pickle")
        print("Checkpoint keys:", checkpoint.keys())
        
        # 详细分析内容
        for key, value in checkpoint.items():
            print(f"\n--- {key} ---")
            if isinstance(value, dict):
                print(f"  Type: dict with {len(value)} keys")
                if key == 'model_state_dict':
                    print("  Model state dict keys (first 10):")
                    for i, model_key in enumerate(list(value.keys())[:10]):
                        tensor_shape = value[model_key].shape if hasattr(value[model_key], 'shape') else 'no shape'
                        print(f"    {i}: {model_key} - {tensor_shape}")
                elif key in ['optimizer_state_dict', 'scheduler_state_dict']:
                    print(f"  Keys: {list(value.keys())}")
                elif key == 'model':  # 旧格式
                    print("  Model keys (first 10):")
                    for i, model_key in enumerate(list(value.keys())[:10]):
                        tensor_shape = value[model_key].shape if hasattr(value[model_key], 'shape') else 'no shape'
                        print(f"    {i}: {model_key} - {tensor_shape}")
            else:
                print(f"  Type: {type(value)}")
                print(f"  Value: {value}")
        
        return checkpoint
        
    except Exception as e:
        print(f"❌ Failed to analyze checkpoint: {e}")
        import traceback
        traceback.print_exc()
        return None

def suggest_fix(checkpoint):
    """根据检查点内容建议修复方法"""
    print("\n" + "="*50)
    print("SUGGESTED FIX")
    print("="*50)
    
    if 'model_state_dict' in checkpoint:
        print("✓ Checkpoint contains 'model_state_dict'")
        print("  Training script should use: checkpoint['model_state_dict']")
    elif 'model' in checkpoint:
        print("✓ Checkpoint contains 'model'")
        print("  Training script should use: checkpoint['model']")
    else:
        print("❓ Unknown checkpoint format")
    
    if 'epoch' in checkpoint:
        print(f"✓ Checkpoint contains epoch: {checkpoint['epoch']}")
    
    if 'optimizer_state_dict' in checkpoint:
        print("✓ Checkpoint contains optimizer state")
    
    if 'scheduler_state_dict' in checkpoint:
        print("✓ Checkpoint contains scheduler state")

def main():
    # 处理命令行参数
    if len(sys.argv) > 1:
        filepath = sys.argv[1]
        # 检查是否是参数标志
        if filepath.startswith('-'):
            print("Usage: python analyze_checkpoint_fixed.py [checkpoint_path]")
            print("If no path provided, uses default path")
            default_path = "outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl"
            print(f"Using default path: {default_path}")
            filepath = default_path
    else:
        filepath = "outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl"
    
    checkpoint = analyze_checkpoint(filepath)
    if checkpoint:
        suggest_fix(checkpoint)

if __name__ == "__main__":
    main()

Usage: python analyze_checkpoint_fixed.py [checkpoint_path]
If no path provided, uses default path
Using default path: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl
Analyzing checkpoint: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl
File size: 592939 bytes
✅ Successfully loaded with pickle
Checkpoint keys: dict_keys(['epoch', 'timestamp', 'metrics', 'additional_info', 'model', 'framework'])

--- epoch ---
  Type: <class 'int'>
  Value: 0

--- timestamp ---
  Type: <class 'str'>
  Value: 2025-10-30T09:20:20.267650

--- metrics ---
  Type: dict with 0 keys

--- additional_info ---
  Type: dict with 0 keys

--- model ---
  Type: dict with 8 keys
  Model keys (first 10):
    0: epoch - no shape
    1: model_state_dict - no shape
    2: optimizer_state_dict - no shape
    3: scheduler_state_dict - no shape
    4: 

In [11]:
# convert_checkpoint_compat_fixed.py
import torch
import pickle
import os
import sys

def convert_checkpoint_for_compatibility(old_path, new_path):
    """转换检查点格式以兼容当前训练脚本"""
    try:
        # 检查原文件是否存在
        if not os.path.exists(old_path):
            print(f"❌ Original checkpoint not found: {old_path}")
            return False
        
        # 加载旧检查点
        with open(old_path, 'rb') as f:
            old_checkpoint = pickle.load(f)
        
        print("Old checkpoint keys:", old_checkpoint.keys())
        
        # 创建兼容格式的检查点
        new_checkpoint = {}
        
        # 提取模型状态
        if 'model_state_dict' in old_checkpoint:
            new_checkpoint['model_state_dict'] = old_checkpoint['model_state_dict']
            print("✓ Using model_state_dict")
        elif 'model' in old_checkpoint:
            new_checkpoint['model_state_dict'] = old_checkpoint['model']
            print("✓ Converted 'model' to 'model_state_dict'")
        else:
            print("❌ No model state found")
            return False
        
        # 提取训练状态
        if 'epoch' in old_checkpoint:
            new_checkpoint['epoch'] = old_checkpoint['epoch']
            print(f"✓ Using epoch: {old_checkpoint['epoch']}")
        
        if 'optimizer_state_dict' in old_checkpoint:
            new_checkpoint['optimizer_state_dict'] = old_checkpoint['optimizer_state_dict']
            print("✓ Using optimizer_state_dict")
        
        if 'scheduler_state_dict' in old_checkpoint:
            new_checkpoint['scheduler_state_dict'] = old_checkpoint['scheduler_state_dict']
            print("✓ Using scheduler_state_dict")
        
        if 'best_metric' in old_checkpoint:
            new_checkpoint['best_metric'] = old_checkpoint['best_metric']
            print(f"✓ Using best_metric: {old_checkpoint['best_metric']}")
        
        if 'best_epoch' in old_checkpoint:
            new_checkpoint['best_epoch'] = old_checkpoint['best_epoch']
            print(f"✓ Using best_epoch: {old_checkpoint['best_epoch']}")
        
        # 保存新格式检查点
        torch.save(new_checkpoint, new_path, pickle_protocol=2)
        print(f"✅ Converted checkpoint saved: {new_path}")
        
        # 验证
        test_checkpoint = torch.load(new_path, map_location='cpu')
        print("New checkpoint keys:", test_checkpoint.keys())
        
        return True
        
    except Exception as e:
        print(f"❌ Conversion failed: {e}")
        import traceback
        traceback.print_exc()
        return False

def main():
    # 处理命令行参数
    if len(sys.argv) > 1:
        old_path = sys.argv[1]
        # 检查是否是参数标志
        if old_path.startswith('-'):
            print("Usage: python convert_checkpoint_compat_fixed.py [old_checkpoint_path] [new_checkpoint_path]")
            print("If no paths provided, uses default paths")
            default_old = "outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl"
            default_new = default_old.replace('.pkl', '_compatible.pth')
            print(f"Using default paths:")
            print(f"  Old: {default_old}")
            print(f"  New: {default_new}")
            old_path = default_old
            new_path = default_new
        else:
            if len(sys.argv) > 2:
                new_path = sys.argv[2]
            else:
                new_path = old_path.replace('.pkl', '_compatible.pth')
    else:
        old_path = "outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl"
        new_path = old_path.replace('.pkl', '_compatible.pth')
    
    print(f"Converting checkpoint for compatibility...")
    print(f"From: {old_path}")
    print(f"To: {new_path}")
    
    success = convert_checkpoint_for_compatibility(old_path, new_path)
    
    if success:
        print(f"\n🎉 Conversion successful!")
        print(f"Now use the converted checkpoint:")
        print(f"python train_chaotic.py ... --resume {new_path}")
    else:
        print(f"\n💥 Conversion failed!")

if __name__ == "__main__":
    main()

Usage: python convert_checkpoint_compat_fixed.py [old_checkpoint_path] [new_checkpoint_path]
If no paths provided, uses default paths
Using default paths:
  Old: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl
  New: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020_compatible.pth
Converting checkpoint for compatibility...
From: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl
To: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020_compatible.pth
Old checkpoint keys: dict_keys(['epoch', 'timestamp', 'metrics', 'additional_info', 'model', 'framework'])
✓ Converted 'model' to 'model_state_dict'
✓ Using epoch: 0
✅ Converted checkpoint saved: outputs/chaotic/

In [12]:
# specialized_checkpoint_fix.py
import torch
import pickle
import os
import sys

def extract_model_from_experiment_checkpoint(old_path, new_path):
    """从实验状态检查点中提取模型状态"""
    try:
        # 检查原文件是否存在
        if not os.path.exists(old_path):
            print(f"❌ Original checkpoint not found: {old_path}")
            return False
        
        # 加载旧检查点
        with open(old_path, 'rb') as f:
            old_checkpoint = pickle.load(f)
        
        print("Old checkpoint keys:", old_checkpoint.keys())
        
        # 提取模型状态
        if 'model_state_dict' in old_checkpoint:
            model_state_dict = old_checkpoint['model_state_dict']
            print("✓ Found model_state_dict")
        elif 'model' in old_checkpoint:
            model_state_dict = old_checkpoint['model']
            print("✓ Found model (converting to model_state_dict)")
        else:
            print("❌ No model state found in checkpoint")
            return False
        
        # 创建新的检查点，只包含模型状态
        new_checkpoint = {
            'model_state_dict': model_state_dict,
            'epoch': old_checkpoint.get('epoch', 0),
            'best_metric': old_checkpoint.get('best_metric', 0.0),
            'best_epoch': old_checkpoint.get('best_epoch', 0)
        }
        
        # 如果有优化器和调度器状态，也保存
        if 'optimizer_state_dict' in old_checkpoint:
            new_checkpoint['optimizer_state_dict'] = old_checkpoint['optimizer_state_dict']
            print("✓ Added optimizer_state_dict")
        
        if 'scheduler_state_dict' in old_checkpoint:
            new_checkpoint['scheduler_state_dict'] = old_checkpoint['scheduler_state_dict']
            print("✓ Added scheduler_state_dict")
        
        # 保存新格式检查点
        torch.save(new_checkpoint, new_path, pickle_protocol=2)
        print(f"✅ Extracted model checkpoint saved: {new_path}")
        
        # 验证
        test_checkpoint = torch.load(new_path, map_location='cpu')
        print("New checkpoint keys:", test_checkpoint.keys())
        print(f"Model state dict keys: {len(test_checkpoint['model_state_dict'])}")
        
        return True
        
    except Exception as e:
        print(f"❌ Extraction failed: {e}")
        import traceback
        traceback.print_exc()
        return False

def main():
    # 处理命令行参数
    if len(sys.argv) > 1:
        old_path = sys.argv[1]
        # 检查是否是参数标志
        if old_path.startswith('-'):
            print("Usage: python specialized_checkpoint_fix.py [old_checkpoint_path] [new_checkpoint_path]")
            print("If no paths provided, uses default paths")
            default_old = "outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl"
            default_new = default_old.replace('.pkl', '_model_only.pth')
            print(f"Using default paths:")
            print(f"  Old: {default_old}")
            print(f"  New: {default_new}")
            old_path = default_old
            new_path = default_new
        else:
            if len(sys.argv) > 2:
                new_path = sys.argv[2]
            else:
                new_path = old_path.replace('.pkl', '_model_only.pth')
    else:
        old_path = "outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl"
        new_path = old_path.replace('.pkl', '_model_only.pth')
    
    print(f"Extracting model from experiment checkpoint...")
    print(f"From: {old_path}")
    print(f"To: {new_path}")
    
    success = extract_model_from_experiment_checkpoint(old_path, new_path)
    
    if success:
        print(f"\n🎉 Extraction successful!")
        print(f"Now use the extracted model checkpoint:")
        print(f"python train_chaotic.py ... --resume {new_path}")
    else:
        print(f"\n💥 Extraction failed!")

if __name__ == "__main__":
    main()

Usage: python specialized_checkpoint_fix.py [old_checkpoint_path] [new_checkpoint_path]
If no paths provided, uses default paths
Using default paths:
  Old: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl
  New: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020_model_only.pth
Extracting model from experiment checkpoint...
From: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl
To: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020_model_only.pth
Old checkpoint keys: dict_keys(['epoch', 'timestamp', 'metrics', 'additional_info', 'model', 'framework'])
✓ Found model (converting to model_state_dict)
✅ Extracted model checkpoint saved: outputs/chaotic/experim

In [17]:
!python simple_checkpoint_test.py

Testing checkpoint: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl
File size: 592939 bytes
❌ torch.load failed: Invalid magic number; corrupt file?
✅ Successfully loaded with pickle
Keys: dict_keys(['epoch', 'timestamp', 'metrics', 'additional_info', 'model', 'framework'])

🎉 Checkpoint test successful!


In [18]:
!python complete_solution.py

COMPLETE CHECKPOINT SOLUTION

1. Checking checkpoint file...
✅ Checkpoint found: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl
   Size: 592939 bytes

2. Testing checkpoint loading...
❌ torch.load failed: Invalid magic number; corrupt file?
✅ Loaded with pickle
✅ Checkpoint keys: dict_keys(['epoch', 'timestamp', 'metrics', 'additional_info', 'model', 'framework'])

3. Extracting model state...
✓ Found model (converting to model_state_dict)
✓ Model state dict has 8 keys

4. Creating compatible checkpoint...
✅ Saved compatible checkpoint: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_compatible.pth

5. Verifying new checkpoint...
✅ Verification successful
   Keys: dict_keys(['model_state_dict', 'epoch', 'best_metric', 'best_epoch', 'training_stats'])
   Epoch: 0
   Best metric: 0.0

🎉 COMPLETE SOLUTION SUCCESSFUL!

Next steps:
1. Use the trai

In [21]:
# test_fix.py
import torch
import os

def test_fix():
    """测试修复后的检查点加载"""
    checkpoint_path = "outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_proper.pth"
    
    if not os.path.exists(checkpoint_path):
        print(f"❌ Checkpoint not found: {checkpoint_path}")
        return False
    
    try:
        # 使用 torch.load 加载 .pth 文件
        checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
        print(f"✅ Successfully loaded with torch.load")
        print(f"Keys: {checkpoint.keys()}")
        
        # 检查必要的键
        required_keys = ['model_state_dict', 'epoch']
        for key in required_keys:
            if key in checkpoint:
                print(f"✓ Required key '{key}' found")
            else:
                print(f"❌ Required key '{key}' missing")
        
        return True
        
    except Exception as e:
        print(f"❌ Failed to load: {e}")
        return False

if __name__ == "__main__":
    success = test_fix()
    
    if success:
        print(f"\n🎉 Fix verified! The checkpoint should work now.")
    else:
        print(f"\n💥 Fix verification failed!")

✅ Successfully loaded with torch.load
Keys: dict_keys(['model_state_dict', 'epoch', 'best_metric', 'best_epoch', 'training_stats'])
✓ Required key 'model_state_dict' found
✓ Required key 'epoch' found

🎉 Fix verified! The checkpoint should work now.


In [20]:
# create_proper_checkpoint.py
import torch
import pickle
import os

def create_proper_checkpoint():
    """创建正确的检查点格式"""
    # 原始检查点路径
    original_pkl = "outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl"
    # 新检查点路径
    new_pth = "outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_proper.pth"
    
    print(f"Creating proper checkpoint format...")
    print(f"From: {original_pkl}")
    print(f"To: {new_pth}")
    
    if not os.path.exists(original_pkl):
        print(f"❌ Original checkpoint not found: {original_pkl}")
        return False
    
    try:
        # 使用pickle加载原始检查点
        with open(original_pkl, 'rb') as f:
            original_checkpoint = pickle.load(f)
        
        print(f"✅ Loaded original checkpoint with pickle")
        print(f"Original keys: {original_checkpoint.keys()}")
        
        # 提取模型状态
        if 'model' in original_checkpoint:
            model_state_dict = original_checkpoint['model']
            print("✓ Found model state in 'model' key")
        else:
            print("❌ No model state found in original checkpoint")
            return False
        
        # 创建标准格式的检查点
        proper_checkpoint = {
            'model_state_dict': model_state_dict,
            'epoch': original_checkpoint.get('epoch', 0),
            'best_metric': original_checkpoint.get('metrics', {}).get('best_val_accuracy', 0.0),
            'best_epoch': original_checkpoint.get('epoch', 0),  # 假设当前epoch就是最佳
            'training_stats': original_checkpoint.get('additional_info', {})
        }
        
        # 使用 torch.save 保存为标准 .pth 格式
        torch.save(proper_checkpoint, new_pth)
        print(f"✅ Saved proper checkpoint: {new_pth}")
        
        # 验证新检查点
        test_checkpoint = torch.load(new_pth, map_location='cpu')
        print(f"✅ Verification successful - keys: {test_checkpoint.keys()}")
        
        return True
        
    except Exception as e:
        print(f"❌ Failed to create proper checkpoint: {e}")
        import traceback
        traceback.print_exc()
        return False

if __name__ == "__main__":
    success = create_proper_checkpoint()
    
    if success:
        print(f"\n🎉 Proper checkpoint created successfully!")
        print(f"Use this checkpoint with:")
        print(f"python train_chaotic.py ... --resume outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_proper.pth")
    else:
        print(f"\n💥 Failed to create proper checkpoint!")

Creating proper checkpoint format...
From: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl
To: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_proper.pth
✅ Loaded original checkpoint with pickle
Original keys: dict_keys(['epoch', 'timestamp', 'metrics', 'additional_info', 'model', 'framework'])
✓ Found model state in 'model' key
✅ Saved proper checkpoint: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_proper.pth
✅ Verification successful - keys: dict_keys(['model_state_dict', 'epoch', 'best_metric', 'best_epoch', 'training_stats'])

🎉 Proper checkpoint created successfully!
Use this checkpoint with:
python train_chaotic.py ... --resume outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_proper.pth


In [22]:
# check_model_compatibility.py
import torch
import pickle
import os

def check_compatibility():
    """检查模型架构兼容性"""
    # 原始检查点
    original_pkl = "outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl"
    # 当前模型检查点
    current_pth = "outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_proper.pth"
    
    print("Checking model architecture compatibility...")
    
    # 加载原始检查点
    with open(original_pkl, 'rb') as f:
        original = pickle.load(f)
    
    # 加载当前检查点
    current = torch.load(current_pth, map_location='cpu')
    
    print(f"Original checkpoint keys: {original.keys()}")
    print(f"Current checkpoint keys: {current.keys()}")
    
    # 获取模型状态字典
    if 'model' in original:
        original_model = original['model']
        print(f"Original model keys: {len(original_model)}")
    else:
        print("❌ No model in original checkpoint")
        return False
    
    if 'model_state_dict' in current:
        current_model = current['model_state_dict']
        print(f"Current model keys: {len(current_model)}")
    else:
        print("❌ No model_state_dict in current checkpoint")
        return False
    
    # 比较键
    original_keys = set(original_model.keys())
    current_keys = set(current_model.keys())
    
    print(f"\n--- Key Comparison ---")
    print(f"Keys in original but not in current: {len(original_keys - current_keys)}")
    print(f"Keys in current but not in original: {len(current_keys - original_keys)}")
    print(f"Common keys: {len(original_keys & current_keys)}")
    
    # 显示不匹配的键
    missing_in_current = original_keys - current_keys
    if missing_in_current:
        print(f"\n--- Missing in current model (first 10) ---")
        for key in list(missing_in_current)[:10]:
            print(f"  {key}")
    
    extra_in_current = current_keys - original_keys
    if extra_in_current:
        print(f"\n--- Extra in current model (first 10) ---")
        for key in list(extra_in_current)[:10]:
            print(f"  {key}")
    
    return True

if __name__ == "__main__":
    check_compatibility()

Checking model architecture compatibility...
Original checkpoint keys: dict_keys(['epoch', 'timestamp', 'metrics', 'additional_info', 'model', 'framework'])
Current checkpoint keys: dict_keys(['model_state_dict', 'epoch', 'best_metric', 'best_epoch', 'training_stats'])
Original model keys: 8
Current model keys: 8

--- Key Comparison ---
Keys in original but not in current: 0
Keys in current but not in original: 0
Common keys: 8


In [23]:
# ultimate_checkpoint_fix.py
import torch
import pickle
import os
import sys

def ultimate_checkpoint_solution():
    """终极检查点解决方案"""
    print("="*70)
    print("ULTIMATE CHECKPOINT RECOVERY SOLUTION")
    print("="*70)
    
    # 定义所有可能的检查点路径
    checkpoint_paths = [
        "outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl",
        "outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251026_015146/checkpoint_epoch_0000_20251027_103724.pkl"
    ]
    
    # 找到实际存在的检查点
    available_checkpoints = []
    for path in checkpoint_paths:
        if os.path.exists(path):
            available_checkpoints.append(path)
            print(f"✅ Found: {path} ({os.path.getsize(path)} bytes)")
    
    if not available_checkpoints:
        print("❌ No checkpoint files found!")
        return None
    
    # 使用第一个可用的检查点
    original_path = available_checkpoints[0]
    ultimate_path = original_path.replace('.pkl', '_ultimate.pth')
    
    print(f"\nUsing checkpoint: {original_path}")
    
    try:
        # 步骤1: 加载原始检查点
        print("\n1. Loading original checkpoint...")
        with open(original_path, 'rb') as f:
            original_checkpoint = pickle.load(f)
        
        print(f"   Original keys: {original_checkpoint.keys()}")
        
        # 步骤2: 提取模型状态
        print("\n2. Extracting model state...")
        if 'model' not in original_checkpoint:
            print("❌ No 'model' key in checkpoint")
            return None
        
        original_model_state = original_checkpoint['model']
        print(f"   Model keys: {len(original_model_state)}")
        
        # 步骤3: 创建当前模型以获取架构
        print("\n3. Creating current model architecture...")
        try:
            from chaotic_network import ChaoticSpeakerRecognitionNetwork
            current_model = ChaoticSpeakerRecognitionNetwork(
                num_speakers=251,
                device='cpu'
            )
            current_model_state = current_model.state_dict()
            print(f"   Current model keys: {len(current_model_state)}")
        except Exception as e:
            print(f"❌ Failed to create current model: {e}")
            return None
        
        # 步骤4: 智能参数迁移
        print("\n4. Migrating parameters intelligently...")
        migrated_state = {}
        match_stats = {
            'exact': 0,
            'shape_match': 0,
            'partial': 0,
            'missed': 0
        }
        
        for current_key, current_tensor in current_model_state.items():
            matched = False
            
            # 策略1: 精确匹配
            if current_key in original_model_state:
                if original_model_state[current_key].shape == current_tensor.shape:
                    migrated_state[current_key] = original_model_state[current_key]
                    match_stats['exact'] += 1
                    matched = True
                    print(f"   ✓ Exact: {current_key}")
            
            # 策略2: 形状匹配但键名不同
            if not matched:
                for orig_key, orig_tensor in original_model_state.items():
                    if (orig_tensor.shape == current_tensor.shape and 
                        any(part in orig_key for part in current_key.split('.'))):
                        migrated_state[current_key] = orig_tensor
                        match_stats['shape_match'] += 1
                        matched = True
                        print(f"   ≈ Shape: {orig_key} -> {current_key}")
                        break
            
            # 策略3: 部分匹配（子字符串）
            if not matched:
                for orig_key in original_model_state.keys():
                    common_parts = set(orig_key.split('.')) & set(current_key.split('.'))
                    if len(common_parts) >= 2:  # 至少有2个共同部分
                        if original_model_state[orig_key].shape == current_tensor.shape:
                            migrated_state[current_key] = original_model_state[orig_key]
                            match_stats['partial'] += 1
                            matched = True
                            print(f"   ~ Partial: {orig_key} -> {current_key}")
                            break
            
            # 策略4: 无法匹配，使用随机初始化
            if not matched:
                migrated_state[current_key] = current_tensor
                match_stats['missed'] += 1
                print(f"   × Missed: {current_key}")
        
        # 步骤5: 创建终极检查点
        print("\n5. Creating ultimate checkpoint...")
        ultimate_checkpoint = {
            'model_state_dict': migrated_state,
            'epoch': original_checkpoint.get('epoch', 0),
            'best_metric': original_checkpoint.get('metrics', {}).get('best_val_accuracy', 0.0),
            'best_epoch': original_checkpoint.get('epoch', 0),
            'migration_stats': match_stats,
            'total_parameters': sum(p.numel() for p in current_model.parameters()),
            'loaded_parameters': sum(p.numel() for p in migrated_state.values() if hasattr(p, 'numel')),
            'original_checkpoint': original_path
        }
        
        # 保存终极检查点
        torch.save(ultimate_checkpoint, ultimate_path)
        print(f"✅ Ultimate checkpoint saved: {ultimate_path}")
        
        # 步骤6: 验证
        print("\n6. Verifying ultimate checkpoint...")
        test_checkpoint = torch.load(ultimate_path, map_location='cpu')
        print(f"   Verified keys: {test_checkpoint.keys()}")
        
        # 加载到模型测试
        try:
            current_model.load_state_dict(test_checkpoint['model_state_dict'], strict=False)
            print("✅ Model loading test: SUCCESS (non-strict)")
        except Exception as e:
            print(f"⚠️  Model loading test: {e}")
        
        # 输出统计信息
        print("\n" + "="*50)
        print("MIGRATION STATISTICS")
        print("="*50)
        total_matched = match_stats['exact'] + match_stats['shape_match'] + match_stats['partial']
        total_keys = len(current_model_state)
        match_rate = total_matched / total_keys
        
        print(f"Exact matches: {match_stats['exact']}/{total_keys}")
        print(f"Shape matches: {match_stats['shape_match']}/{total_keys}") 
        print(f"Partial matches: {match_stats['partial']}/{total_keys}")
        print(f"Missed keys: {match_stats['missed']}/{total_keys}")
        print(f"Overall match rate: {match_rate:.1%}")
        
        if match_rate >= 0.7:
            print("🎉 EXCELLENT: High compatibility, should work well!")
        elif match_rate >= 0.3:
            print("👍 GOOD: Moderate compatibility, should provide good initialization")
        else:
            print("⚠️  LOW: Limited compatibility, consider training from scratch")
        
        return ultimate_path
        
    except Exception as e:
        print(f"❌ Ultimate solution failed: {e}")
        import traceback
        traceback.print_exc()
        return None

if __name__ == "__main__":
    result_path = ultimate_checkpoint_solution()
    
    print("\n" + "="*70)
    if result_path:
        print("🎉 ULTIMATE SOLUTION SUCCESSFUL!")
        print(f"\nUse this checkpoint:")
        print(f"python train_chaotic.py ... --resume {result_path}")
        print(f"\nTraining command:")
        print(f"python train_chaotic.py --system lorenz --epochs 100 --save_models --model_types full_chaotic --data_dir /scratch/project_2003370/yueyao/dataset/train-clean-100/LibriSpeech/train-clean-100 --resume {result_path}")
    else:
        print("💥 ULTIMATE SOLUTION FAILED!")
        print("\nRecommendation: Train from scratch with adjusted learning rate")

ULTIMATE CHECKPOINT RECOVERY SOLUTION
✅ Found: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl (592939 bytes)
✅ Found: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251026_015146/checkpoint_epoch_0000_20251027_103724.pkl (594297 bytes)

Using checkpoint: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_epoch_0000_20251030_092020.pkl

1. Loading original checkpoint...
   Original keys: dict_keys(['epoch', 'timestamp', 'metrics', 'additional_info', 'model', 'framework'])

2. Extracting model state...
   Model keys: 8

3. Creating current model architecture...
❌ Failed to create current model: No module named 'chaotic_network'

💥 ULTIMATE SOLUTION FAILED!

Recommendation: Train from scratch with adjusted learning rate


In [1]:
!python fix_checkpoint.py

🎯 Creating compatible checkpoint...
1. Loading original checkpoint...
   Original keys: dict_keys(['epoch', 'timestamp', 'metrics', 'additional_info', 'model', 'framework'])
   Model parameters: 8
2. Creating compatible checkpoint...
✅ Compatible checkpoint saved: outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_compatible.pth
✅ Verification successful - keys: dict_keys(['model_state_dict', 'epoch', 'best_metric', 'best_epoch'])

🎉 SUCCESS! Use this command to resume training:
python train_chaotic.py --system lorenz --epochs 100 --save_models --model_types full_chaotic --data_dir /scratch/project_2003370/yueyao/dataset/train-clean-100/LibriSpeech/train-clean-100 --resume outputs/chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20251029_151635/checkpoint_compatible.pth
